# Notebook 5 · What do positive and negative Oddbox reviews talk about?
**NTU MSc Growth Hacking | Customer Insight Lab**

The founder wants a claim for the landing page. If positive reviews praise the deliveries, what happens when you read negative reviews about delivery too? Use both groups to find a strength, its limits and an idea worth testing.

Notebook 3 decided which reviews counted as positive or negative. Notebook 4 showed how to name a topic. Here you fit each model once across both groups, then compare how often the same topics appear in each group. Work through the cells in order.

## 1 · Bring in the reviews you selected
Upload the ZIP downloaded from Notebook 3, or select its positive and negative CSV files together. The cell reads the two review groups and leaves out the audit file, which contains overlapping rows. Read the selection rule and rating filter printed below. Who could have been left out?

In [ ]:
import io
import re
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
pd.set_option('display.max_colwidth', None)

uploaded = files.upload()
inputs = uploaded
if len(uploaded) == 1 and next(iter(uploaded)).lower().endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        inputs = {name: archive.read(name) for name in archive.namelist()
                  if name.lower().endswith('.csv') and not name.startswith('__MACOSX/')}
frames = []
for group in ['positive','negative']:
    matches = [name for name in inputs if re.search(r'(?<![a-z])'+group+r'(?![a-z])', Path(name).stem.lower())
               and name.lower().endswith('.csv') and 'audit' not in name.lower()]
    if len(matches) != 1:
        raise ValueError(f'Choose exactly one {group} review CSV from the same Notebook 3 run.')
    frame = pd.read_csv(io.BytesIO(inputs[matches[0]]), dtype=str, keep_default_na=False)
    if 'review_text' not in frame:
        raise ValueError('Use the Notebook 3 export containing review_text.')
    if frame.empty:
        raise ValueError(f'The {group} export is empty. Revisit the star filter and export rule in Notebook 3.')
    if 'export_group' in frame and not frame.export_group.eq(group).all():
        raise ValueError('The group in the filename differs from the labels inside the file.')
    frames.append(frame.assign(group=group, input_file=Path(matches[0]).name))
raw = pd.concat(frames, ignore_index=True)
fields = [k for k in ['export_rule','export_method','selected_ratings'] if k in raw]
display(raw.groupby(['group']+fields, dropna=False).size().rename('imported reviews').to_frame())
if fields and len(raw[fields].drop_duplicates()) > 1:
    raise ValueError('The files record different selection settings. Upload the pair from one run.')

## 2 · Check the evidence
A blank review has no words to model. A repeated import would give one review extra influence. We use the original source row, or the review ID if needed, to recognise repeated copies. When both are missing, identical wording alone is too little evidence to delete a row. Keep the original wording for checking each interpretation.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation, NMF
data = raw.copy()
data['review'] = data.review_text.str.strip()
source = data.get('source_row', pd.Series('', index=data.index)).str.strip()
review_id = data.get('review_id', pd.Series('', index=data.index)).str.strip()
data['key'] = np.select([source.ne(''), review_id.ne('')], ['row-'+source,'id-'+review_id], default='')
known = data.loc[data.key.ne('')]
if known.groupby('key').group.nunique().gt(1).any():
    raise ValueError('A source review appears in both groups. Upload the two exports from one run.')
blank = data.review.eq('')
repeat = ~blank & data.key.ne('') & data.duplicated(['key','group','review'])
print(f'{blank.sum()} blank rows and {repeat.sum()} repeated imports removed.')
data = data.loc[~(blank|repeat)].reset_index(drop=True)
data['id'] = ['R'+str(i+1) for i in data.index]
display(data.groupby('group').head(3)[['group','id','review']])

### Choose how to prepare the words
Start with the original word forms. Stemming trims endings by rule. Lemmatisation uses grammar and a dictionary to recover a base form. Both can bring related words together. Read a prepared review alongside the original before deciding whether the change helps you recognise its subject.

In [ ]:
word_form = "Keep word forms" #@param ["Keep word forms", "Stemming", "Lemmatisation"]
stop = set(ENGLISH_STOP_WORDS) - {'no','not','never'}
if word_form == 'Stemming':
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
elif word_form == 'Lemmatisation':
    import nltk
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet', quiet=True)
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    lemma = WordNetLemmatizer()

def prepare(text):
    words = re.findall(r"[a-z]+", text.lower())
    if word_form == 'Lemmatisation':
        pos_map = {'N':'n', 'V':'v', 'J':'a', 'R':'r'}
        return ' '.join(lemma.lemmatize(w, pos_map.get(tag[0], 'n'))
                        for w, tag in nltk.pos_tag(words) if w not in stop)
    words = [w for w in words if w not in stop]
    if word_form == 'Stemming':
        words = [stemmer.stem(w) for w in words]
    return ' '.join(words)

data['prepared'] = data.review.map(prepare)
display(data[['id','review','prepared']].head(6))

## 3 · How does a review become a row of numbers?
Each row represents one review. Each column represents a word. A count of 1 under “driver” means the word appeared once in that review. Look at the first two rows. Which columns would help you tell their subjects apart?

**LDA** will use these counts. **NMF** will use a weighted version called **TF–IDF**. It gives relatively more weight to words that distinguish one review from others. For example, if every review mentioned “box”, that word alone would do little to separate delivery comments from cooking comments. Compare numbers within each table. The tables use different scales.

We also count pairs of neighbouring words in the prepared text. Removing common words can bring two words together, so check the original review before quoting a phrase.

In [ ]:
vectoriser = CountVectorizer(ngram_range=(1, 2))
usable_words = data.prepared.str.contains(r'\b\w\w+\b')
print(f'{(~usable_words).sum()} reviews have no usable terms after preparation.')
model_data = data.loc[usable_words].reset_index(drop=True)
if len(model_data) < 2:
    raise ValueError('Too few reviews have usable words. Check your text column.')
if model_data.group.nunique() < 2:
    raise ValueError('One group has no usable words. Check your source files or preparation choice.')
display(model_data.group.value_counts().rename('modelled reviews').to_frame())
counts = vectoriser.fit_transform(model_data.prepared)
terms = vectoriser.get_feature_names_out()
weighted = TfidfTransformer().fit_transform(counts)
# Show words actually found in the first two reviews.
visible = np.flatnonzero(counts[:2].sum(axis=0).A1)[:12]
for title, matrix in [('Word counts', counts), ('TF–IDF weights', weighted)]:
    print(title)
    display(pd.DataFrame(matrix[:2, visible].toarray(), columns=terms[visible],
                         index=model_data.id[:2]).round(2))

## 4 · Discover topics shared across the two groups
LDA uses word counts to estimate the mixture of topics in each review. NMF uses TF–IDF weights to find recurring word patterns. The groups share one vocabulary and one fitted model per method. This makes a positive review’s LDA topic 1 directly comparable with a negative review’s LDA topic 1. NMF learns its own topics, so match across methods by wording rather than number.

Read the ranked words and two-word phrases beside each cloud. Check the example review before giving the topic a short name. If one group is much larger, its language may shape the topics more strongly.

In [ ]:
number_of_topics = 3 #@param [2,3,4] {type:"raw"}
if number_of_topics > min(counts.shape):
    raise ValueError('Choose fewer topics or upload more reviews with distinct words.')
models = {
    'LDA': LatentDirichletAllocation(n_components=number_of_topics, random_state=42, max_iter=40),
    'NMF': NMF(n_components=number_of_topics, init='nndsvda', random_state=42, max_iter=800)
}
shares = {}
for method, model in models.items():
    strength = model.fit_transform(counts if method == 'LDA' else weighted)
    totals = strength.sum(axis=1, keepdims=True)
    shares[method] = np.divide(strength, totals, out=np.zeros_like(strength), where=totals>0)

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.transforms import Bbox

def topic_page(method, i, label):
    weights = models[method].components_[i]
    order = np.argsort(-weights, kind='stable')
    top = [j for j in order if weights[j] > 0][:12]
    fig = plt.figure(figsize=(11.7, 8.3))
    fig.suptitle(f'{method} topic {i+1}  |  {label}', fontsize=16, wrap=True)
    cloud = fig.add_axes([.025, .20, .52, .65], xlim=(0,1), ylim=(0,1))
    cloud.axis('off')
    table = fig.add_axes([.56, .16, .42, .69])
    table.axis('off')
    table.set_title('Ranked words and two-word phrases', loc='left')
    renderer = fig.canvas.get_renderer()
    occupied = []
    ranked = []
    for rank, j in enumerate(top, 1):
        term = terms[j]
        share = 100 * weights[j] / weights.sum()
        ranked.append([rank, term, f'{share:.2f}%'])
        size = 12 + 27 * np.sqrt(weights[j] / weights[top[0]])
        artist = cloud.text(.5,.5,term,ha='center',va='center',fontsize=size,
                            color=plt.cm.viridis(.18+.55*(rank-1)/12))
        for attempt in range(1500):
            radius = .013 * np.sqrt(attempt)
            angle = attempt * 2.4
            artist.set_position((.5+radius*np.cos(angle),.5+radius*np.sin(angle)))
            box = artist.get_window_extent(renderer).expanded(1.04,1.08)
            if cloud.bbox.contains(*box.min) and cloud.bbox.contains(*box.max) and not any(box.overlaps(b) for b in occupied):
                occupied.append(box)
                break
            if attempt in (500,1000):
                artist.set_fontsize(artist.get_fontsize()*.78)
        else:
            artist.remove()
    if ranked:
        table.table(cellText=ranked,colLabels=['Rank','Term','Topic weight'],cellLoc='left',
                    colWidths=[.13,.61,.26],bbox=[0,.04,1,.92])
    fig.text(.04,.10,'Topic weight = this term’s share of the total weight across every term in this topic.')
    fig.text(.04,.06,'Only the leading terms are shown. Compare these with the wording of the original reviews.')
    return fig

for method in models:
    for i in range(number_of_topics):
        fig = topic_page(method,i,'What would you call this topic?')
        plt.show()
        plt.close(fig)
        best = int(shares[method][:,i].argmax())
        print(f'Example {model_data.id[best]} | {model_data.review[best]}')


**Name each topic.** Read the numbered list alongside each cloud. A term with 4% accounts for four hundredths of this topic’s total term weight. This differs from the percentage of a review assigned to a topic. Pick two high-ranking terms and explain how the example review links them. Then read its example review. “Doorstep delivery” is a more useful name than “negative” if the evidence is about where the driver left the box. Enter your own names when prompted. If the same subject appears in both models, use the same name so you can compare them.

In [ ]:
names = {}
for method in models:
    names[method] = []
    for i in range(number_of_topics):
        default = f'{method} topic {i+1}'
        names[method].append(input(f'Name {default}: ').strip() or default)

## 5 · Which topics matter in each group?
Pick one review from the preview. Predict its main subject, then inspect its full topic profile. LDA gives estimated topic shares. NMF percentages show how much of the review’s total fitted weight belongs to each pattern. A narrow lead between two topics is a reason to read the review again.

In [ ]:
display(model_data.groupby('group').head(3)[['id','group','review']])
review_id = input('Enter a displayed review ID, or press Return for the first review in the file: ').strip() or model_data.id.iloc[0]
matches = model_data.index[model_data.id.eq(review_id.strip().upper())]
if len(matches) != 1:
    raise ValueError('Choose an ID from the displayed reviews.')
row = matches[0]
print(model_data.review[row])
output = model_data.copy()
for method in models:
    weights = shares[method]
    for i, name in enumerate(names[method]):
        output[f'{method} {i+1} {name} (%)'] = (100*weights[:,i]).round(1)
    output[method+' main topic'] = [names[method][v.argmax()] if v.sum() else 'No matching words' for v in weights]
    table = pd.DataFrame({'topic':names[method], 'share (%)':(100*weights[row]).round(1)})
    print(method)
    display(table.sort_values('share (%)',ascending=False))
display(output[['id','review'] + [col for col in output if col.endswith('(%)')]].head(12))

summary = []
examples = []
for method, weights in shares.items():
    for group, part in model_data.groupby('group'):
        scores = weights[part.index]
        has_signal = scores.sum(axis=1) > 0
        for i, name in enumerate(names[method]):
            main = int(((scores.argmax(axis=1)==i) & has_signal).sum())
            summary.append([method,i+1,name,group,len(part),main,100*main/len(part),
                            100*scores[has_signal,i].mean() if has_signal.any() else np.nan])
            best = scores[:,i].argmax()
            if scores[best,i] > 0:
                item = part.iloc[best]
                examples.append([method,i+1,name,group,item.id,item.review,100*scores[best,i]])
summary = pd.DataFrame(summary,columns=['method','topic','name','group','group size','main topic count',
                                         'main topic %','average share among scored reviews %'])
examples = pd.DataFrame(examples,columns=['method','topic','name','group','id','review','topic %'])
display(summary.pivot(index=['method','topic','name'],columns='group',values=['group size','main topic count','main topic %']).round(1))
display(examples.sort_values(['method','topic','group']).round(1))

## 6 · What would you tell the founder?
Read **main topic %** as the share of reviews in that group for which this was the strongest topic. For example, 20 out of 100 positive reviews and 6 out of 10 negative reviews would give 20% and 60%. More positive reviewers mentioned it as their main topic in absolute numbers, while it was more common within the smaller negative group.

1. Pick one topic. Report the counts and percentages in each group using the same model.
2. Read one example from each group. What exactly did one customer value and the other criticise? A topic can have the same name while the experiences differ.
3. Check the second method. Can you find a similar topic by reading its phrases? If the methods differ, explain how that affects your proposed claim.
4. Write a test for the founder. “We would test ___ because these reviewers described ___. We would measure ___.” Explain whose views the Notebook 3 selection rule may have missed.

The final ZIP contains the labelled cloud PDF, the review-level topic percentages, the group comparison and the example reviews.

In [ ]:
from google.colab import files
output['word_form'] = word_form
output['number_of_topics'] = number_of_topics
filename_export = 'Notebook-5-review-topics.csv'
output.to_csv(filename_export, index=False, encoding='utf-8-sig')
pdf_name = 'Notebook-5-topic-clouds.pdf'
with PdfPages(pdf_name) as pdf:
    for method in models:
        for i, label in enumerate(names[method]):
            fig = topic_page(method, i, label)
            pdf.savefig(fig)
            plt.close(fig)
import zipfile
download = 'Notebook-5-topic-results.zip'
with zipfile.ZipFile(download, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.writestr('group-comparison.csv', summary.round(3).to_csv(index=False).encode('utf-8-sig'))
    bundle.writestr('example-reviews.csv', examples.round(3).to_csv(index=False).encode('utf-8-sig'))
    bundle.write(pdf_name)
    bundle.write(filename_export)
files.download(download)